# Offline replay activity of the animals exposed to strong aversive stimuli (reported in Wu et al.)

### Import

In [1]:
import os, sys, pickle, warnings
import numpy as np
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

base_path = os.path.sep.join(os.path.abspath("__file__").split(os.path.sep)[:-2])
sys.path.insert(0, os.path.join(base_path, "code/functions"))

from linear_shock_functions import data_path, load_PF_starts, retreive_ID_from_position
from stat_functions import *

plot_path = os.path.join(data_path, "plots")
os.makedirs(plot_path, exist_ok=True)

save_plot = False

In [ ]:
from global_variables import cell_per_unit
from linear_shock_variables import num_state_total, num_CA3_neurons, rest_time

cue_state = 3
sim_duration = rest_time*2   # ms
trial_number = 10
trial_dirs = [os.path.join(data_path, "trial%d" % i) for i in range(trial_number)]

fatigue_state = np.arange(cue_state)   # states 1-3 neurons are fatigued
sim_duration  = rest_time*2   # ms

PF_dict = load_PF_starts()

condition_fname = [
    "CA3_replay_wu_fatigue.npz",
    "CA3_replay_wu_ctrl.npz",
]

### SFig. 5b, Juxtaposed raster: fatigued (top) vs. control (bottom)

In [8]:
targ_trial = 2

In [ ]:
def wu_state_color(s):
    if np.isin(s, fatigue_state):                return "red"
    if s == cue_state:                           return "orange"
    if s > cue_state:                            return "steelblue"   # forward
    return "tomato"                                                     # backward

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 3),
    sharey=True,
    gridspec_kw={"hspace": 0.06, "wspace": 0.05},
)

trial_idx = targ_trial; exp_dir = trial_dirs[trial_idx]

for row, fname in enumerate(condition_fname):
    ax = axes[row]

    d          = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
    spike_t    = d["spike_times_CA3_PC"]           # ms
    spike_nids = d["spiking_neurons_CA3_PC"].astype(int)
    states_v   = np.array([min(int(n) // cell_per_unit, num_state_total-1) for n in spike_nids])

    for s in range(num_state_total):
        mask = states_v == s
        if mask.any():
            ax.scatter(
                spike_t[mask], spike_nids[mask],
                s=0.3, c=wu_state_color(s), rasterized=True, marker='.',
                zorder=3 if s in fatigue_state+[cue_state] else
                        2 if s > cue_state else 1,
            )

    # state boundary lines
    for y0 in np.arange(num_state_total) * cell_per_unit:
        ax.axhline(y0, color="black", linewidth=0.4, linestyle="--", alpha=0.35)

    ax.set_xlim(0, sim_duration)
    ax.set_ylim(0, num_CA3_neurons)
    ax.tick_params(axis="x", labelsize=6)

    if row == 0:
        ax.set_title(f"E{trial_idx}", fontsize=9, pad=2)
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Time (ms)", fontsize=7)

    if trial_idx == 0:
        ax.set_ylabel(ROW_TITLES_WU[row] + "\nneuron ID", fontsize=8)
    else:
        ax.tick_params(labelleft=False)

ytick_pos = np.arange(num_state_total)*cell_per_unit+cell_per_unit//2
for row in range(2):
    ax0 = axes[row]
    ax0.set_yticks(ytick_pos)
    ax0.set_yticklabels([f"S{s+1}" for s in range(num_state_total)], fontsize=6)
    for tick, s in zip(ax0.get_yticklabels(), range(num_state_total)):
        tick.set_color(wu_state_color(s))

plt.show()
plt.savefig(os.path.join(plot_path, "wu_juxtaposed_rasters_trial%d.pdf" % targ_trial),
            bbox_inches="tight")


## Neuron firing rate analyses: forward (→ S8) vs. backward (→ S1)

### SFig. 5c and d, Overall neuron activity and comparison between forward vs. backward activities

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(5, 3))

x_exp   = np.arange(trial_number)
width   = 0.35
COLORS  = {"forward": "steelblue", "backward": "tomato", "unclear": "lightgrey"}
COND_EDGE = ["black", "dimgrey"]

ahead_rate  = np.zeros(trial_number); behind_rate = np.zeros(trial_number)

ax = axs[0]
for trial in range(trial_number):
    exp_dir = trial_dirs[trial]
    temp_f = dict(np.load(os.path.join(exp_dir, condition_fname[0])))
    spike_nids = temp_f["spiking_neurons_CA3_PC"].astype(int)

    # Count spikes per state
    counts = np.zeros(num_state_total, dtype=int)
    for nid in spike_nids:
        pos = PF_dict[nid]
        (s,_) = retreive_ID_from_position(pos)
        if 0 <= s < num_state_total:
            counts[s] += 1

    ahead_rate[trial]  = counts[cue_state + 1:].sum()/(cell_per_unit * (num_state_total-cue_state-1) * sim_duration/1000)   # spikes per neuron per second
    behind_rate[trial] = counts[:cue_state].sum()/(cell_per_unit * cue_state * sim_duration/1000)    # spikes per neuron per second

jitter = 0.04 * (np.random.default_rng(7).random(trial_number) - 0.5)
ax.scatter(np.zeros(trial_number) + jitter, behind_rate, color="tomato",      s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, ahead_rate, color="steelblue", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [behind_rate[i], ahead_rate[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [behind_rate.mean(), ahead_rate.mean()],
            yerr=[behind_rate.std() / np.sqrt(trial_number),
                  ahead_rate.std()  / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Behind", "Ahead"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("Firing rate")
ax.set_title("SZ vs. non-SZ replay", fontsize=10)

ax = axs[1]

behind_rate_ctrl = np.full(trial_number, np.nan)
for trial in range(trial_number):
    exp_dir   = trial_dirs[trial]
    temp_f = dict(np.load(os.path.join(exp_dir, condition_fname[0])))
    ctrl_f = np.load(os.path.join(exp_dir, condition_fname[1]), allow_pickle=True)
    spike_nids = ctrl_f["spiking_neurons_CA3_PC"].astype(int)

    counts = np.zeros(num_state_total, dtype=int)
    for nid in spike_nids:
        pos = PF_dict[nid]
        (s,_) = retreive_ID_from_position(pos)
        if 0 <= s < num_state_total:
            counts[s] += 1

    behind_rate_ctrl[trial] = counts[:cue_state].sum()/(cell_per_unit * cue_state * sim_duration/1000)   # spikes per neuron per second

ax.scatter(np.zeros(trial_number) + jitter, behind_rate_ctrl, color="grey",   s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, behind_rate,      color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [behind_rate_ctrl[i], behind_rate[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [np.nanmean(behind_rate_ctrl), behind_rate.mean()],
            yerr=[np.nanstd(behind_rate_ctrl) / np.sqrt(trial_number),
                  behind_rate.std()           / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Fatigued"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("Firing rate")
ax.set_title("Behind-state replay: fatigued vs. control", fontsize=10)

plt.tight_layout()
plt.show()

if save_plot:
    plt.savefig(os.path.join(plot_path, "wu_FR_comparison.svg"),format='svg',
                bbox_inches="tight")

metrics = [
    ('Spike rate-behind vs. ahead',   behind_rate,  ahead_rate),
    ('Spike rate-without vs. with adaptation',   behind_rate_ctrl,  behind_rate),
]

for label, x, y in metrics:
    # drop NaN seeds for bias comparison
    mask = ~(np.isnan(x) | np.isnan(y))
    xm, ym = x[mask], y[mask]
    t_stat, p_val = stats.ttest_rel(xm, ym)
    d_z = cohens_d_paired(xm, ym)
    ci_lo, ci_hi = ci_paired(xm, ym)
    print(f'{label:20s}: BW={xm.mean():.3f}±{xm.std(ddof=1):.3f}  '
          f'Shock={ym.mean():.3f}±{ym.std(ddof=1):.3f}  '
          f'\nt={t_stat:.3f}  p={p_val:.4f}  d_z={d_z:.3f}  '
          f'CI=[{ci_lo:.3f}, {ci_hi:.3f}]')


## Replay direction analysis: forward (→ S8) vs. backward (→ S1)

In [ ]:
def classify_direction(burst_spike_t, burst_nids, 
                       r_thresh=0.3, min_spikes=5):
    """
    Pearson r between spike time and PF column.
    Returns 'forward', 'backward', or 'unclear'.
    """
    pf_cols = np.array([PF_dict.get(int(n), [0, -1])[1] for n in burst_nids])
    valid   = pf_cols >= 0
    if valid.sum() < min_spikes:
        return "unclear"
    r = np.corrcoef(burst_spike_t[valid], pf_cols[valid])[0, 1]
    if np.isnan(r):  return "unclear"
    if r >  r_thresh: return "forward"
    if r < -r_thresh: return "backward"
    return "unclear"

def count_replay_directions(spike_t_ms, spike_nids, rate_arr, 
                            thresh_factor=1.0, min_dur_ms=100,
                            r_thresh=0.2, min_spikes=5):
    """
    Detect bursts in rate_arr, classify each burst's replay direction.
    Returns dict with counts for 'forward', 'backward', 'unclear'.
    """
    from common_functions import slice_high_activity
    threshold = rate_arr.mean() + thresh_factor * rate_arr.std()
    bursts = slice_high_activity(rate=rate_arr, th=threshold,
                                 min_len=min_dur_ms, len_sim=sim_duration)
    counts = {"forward": 0, "backward": 0, "unclear": 0}
    times = {"forward": [], "backward": [], "unclear": []}
    for t0, t1 in bursts:
        mask = (spike_t_ms >= t0) & (spike_t_ms <= t1)
        if mask.sum() < min_spikes:
            continue
        direction = classify_direction(spike_t_ms[mask], spike_nids[mask],
                                       r_thresh, min_spikes)
        counts[direction] += 1
        times[direction].append((t0, t1))
    counts["n_bursts"] = len(bursts)
    return counts, times

In [ ]:
COND_LABELS = ["Fatigued", "Control"]

direction_labels = ["forward", "backward", "unclear"]
replay_dir_counts = np.zeros((2, trial_number, 3), dtype=int)

for cond_idx, (cond_label, fname) in enumerate(zip(COND_LABELS, condition_fname)):
    print(f"\n=== {cond_label} ===")
    for trial_idx in range(trial_number):
        exp_dir = trial_dirs[trial_idx]

        d          = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
        spike_t    = d["spike_times_CA3_PC"]           # ms
        spike_nids = d["spiking_neurons_CA3_PC"].astype(int)
        rate_arr   = d["rate_CA3_PC"]                  # Hz, shape (T,)

        counts, times = count_replay_directions(spike_t, spike_nids, rate_arr)
        for di, dlabel in enumerate(direction_labels):
            replay_dir_counts[cond_idx, trial_idx, di] = counts[dlabel]

        print(f"  Exposure {trial_idx}: bursts={counts['n_bursts']}  "
              f"fwd={counts['forward']}  bwd={counts['backward']}  "
              f"unclear={counts['unclear']}")
        foldername = "trial%d"%trial_idx
        save_path = os.path.join(data_path,foldername)
        np.savez_compressed(os.path.join(save_path, f"CA3_replay_{COND_LABELS[cond_idx]}_type_{trial_idx}.npz"), times=times, counts=counts)

print("\nDone.")
print("\nForward counts — fatigued:", replay_dir_counts[0, :, 0])
print("Forward counts — control: ", replay_dir_counts[1, :, 0])
print("Backward counts — fatigued:", replay_dir_counts[0, :, 1])
print("Backward counts — control: ", replay_dir_counts[1, :, 1])


=== Fatigued ===
  Exposure 0: bursts=5  fwd=5  bwd=0  unclear=0
  Exposure 1: bursts=4  fwd=4  bwd=0  unclear=0
  Exposure 2: bursts=5  fwd=4  bwd=1  unclear=0
  Exposure 3: bursts=2  fwd=2  bwd=0  unclear=0
  Exposure 4: bursts=6  fwd=6  bwd=0  unclear=0
  Exposure 5: bursts=6  fwd=6  bwd=0  unclear=0
  Exposure 6: bursts=5  fwd=4  bwd=1  unclear=0
  Exposure 7: bursts=5  fwd=5  bwd=0  unclear=0
  Exposure 8: bursts=5  fwd=5  bwd=0  unclear=0
  Exposure 9: bursts=3  fwd=3  bwd=0  unclear=0

=== Control ===
  Exposure 0: bursts=5  fwd=5  bwd=0  unclear=0
  Exposure 1: bursts=6  fwd=3  bwd=3  unclear=0
  Exposure 2: bursts=7  fwd=2  bwd=5  unclear=0
  Exposure 3: bursts=5  fwd=2  bwd=3  unclear=0
  Exposure 4: bursts=9  fwd=3  bwd=6  unclear=0
  Exposure 5: bursts=6  fwd=5  bwd=1  unclear=0
  Exposure 6: bursts=6  fwd=5  bwd=1  unclear=0
  Exposure 7: bursts=7  fwd=7  bwd=0  unclear=0
  Exposure 8: bursts=5  fwd=5  bwd=0  unclear=0
  Exposure 9: bursts=7  fwd=7  bwd=0  unclear=0

Done

### SFig. 5e and f, Total replay count and reply count per direction

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

x_exp   = np.arange(trial_number)
width   = 0.35
COLORS  = {"forward": "steelblue", "backward": "tomato", "unclear": "lightgrey"}
COND_EDGE = ["black", "dimgrey"]

ax = axes[0]
total_fat  = np.maximum(replay_dir_counts[0, :, 0] + replay_dir_counts[0, :, 1], 1)
total_ctrl = np.maximum(replay_dir_counts[1, :, 0] + replay_dir_counts[1, :, 1], 1)

jitter = 0.04 * (np.random.default_rng(7).random(trial_number) - 0.5)
ax.scatter(np.zeros(trial_number) + jitter, total_ctrl, color="grey", s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, total_fat,  color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [total_ctrl[i], total_fat[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [total_ctrl.mean(), total_fat.mean()],
            yerr=[total_ctrl.std() / np.sqrt(trial_number),
                  total_fat.std()  / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Fatigued"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 15); ax.set_yticks(np.arange(0, 16, 5))
ax.set_ylabel("Replay count")
ax.set_title("Total replay count")

ax = axes[1]
bias_fat   = replay_dir_counts[0, :, 0] / total_fat
bias_ctrl  = replay_dir_counts[1, :, 0] / total_ctrl

jitter = 0.1 * (np.random.default_rng(7).random(trial_number) - 0.5)
ax.scatter(np.zeros(trial_number) + jitter, bias_ctrl, color="grey",      s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, bias_fat,  color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [bias_ctrl[i], bias_fat[i]],
            color="lightgrey", linewidth=1, zorder=1)
ax.errorbar([0, 1],
            [bias_ctrl.mean(), bias_fat.mean()],
            yerr=[bias_ctrl.std() / np.sqrt(trial_number),
                  bias_fat.std()  / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=10, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Fatigued"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.1, 1.1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("Forward / (Forward + Backward)")
ax.set_title("Forward bias (paired)")

plt.tight_layout()
plt.savefig(os.path.join(plot_path, "wu_replay_direction_comparison.svg"),format='svg',
            bbox_inches="tight")
plt.show()

metrics = [
    ('Total count', replay_dir_counts[0].sum(-1), replay_dir_counts[1].sum(-1)),
    ('SZ count', replay_dir_counts[0, :, 0], replay_dir_counts[1, :, 0]),
    ('non-SZ count', replay_dir_counts[0, :, 1], replay_dir_counts[1, :, 1]),
]

for label, x, y in metrics:
    mask = ~(np.isnan(x) | np.isnan(y))
    xm, ym = x[mask], y[mask]
    t_stat, p_val = stats.ttest_rel(xm, ym)
    d_z = cohens_d_paired(xm, ym)
    ci_lo, ci_hi = ci_paired(xm, ym)
    print(f'{label}: Control={xm.mean():.3f}±{xm.std(ddof=1):.3f}  '
          f'Fatigued={ym.mean():.3f}±{ym.std(ddof=1):.3f}  '
          f'\nt={t_stat:.3f}  p={p_val:.4f}  d_z={d_z:.3f}  '
          f'CI=[{ci_lo:.3f}, {ci_hi:.3f}]')


Total count: Control=4.600±1.265  Fatigued=6.300±1.252  
t=-3.791  p=0.0043  d_z=-1.199  CI=[-2.714, -0.686]
SZ count: Control=4.400±1.265  Fatigued=4.400±1.838  
t=0.000  p=1.0000  d_z=0.000  CI=[-1.431, 1.431]
non-SZ count: Control=0.200±0.422  Fatigued=1.900±2.234  
t=-2.486  p=0.0347  d_z=-0.786  CI=[-3.247, -0.153]
